### Download the TFLite model

First, we need to download the TFLite model file using the `download_testdata` utility function.

In [6]:
import os
import urllib.request

def download_testdata(url, filename, module):
    module_path = os.path.join('/tmp', module)
    os.makedirs(module_path, exist_ok=True)
    filepath = os.path.join(module_path, filename)
    if not os.path.exists(filepath):
        print(f'Downloading {filename} from {url}...')
        urllib.request.urlretrieve(url, filepath)
        print(f'Downloaded {filename} to {filepath}')
    else:
        print(f'{filename} already exists at {filepath}')
    return filepath

MODEL_URL = "https://github.com/mlcommons/tiny/raw/bceb91c5ad2e2deb295547d81505721d3a87d578/benchmark/training/visual_wake_words/trained_models/vww_96_int8.tflite"
MODEL_NAME = "vww_96_int8.tflite"
MODEL_PATH = download_testdata(MODEL_URL, MODEL_NAME, module="model")

vww_96_int8.tflite already exists at /tmp/model/vww_96_int8.tflite


In [7]:
interpreter = tf.lite.Interpreter(model_path=MODEL_PATH)
interpreter.allocate_tensors()

# Lire les paramètres de quantification par couche
print(f"{'Index':>6}  {'Nom':45s}  {'Scale':>12}  {'Zero':>6}  {'ap_fixed suggéré'}")
print("-" * 85)
for detail in interpreter.get_tensor_details():
    qp = detail['quantization_parameters']
    if len(qp['scales']) > 0:
        scale     = float(qp['scales'][0])
        zp        = int(qp['zero_points'][0])
        # Dériver I depuis le scale TFLite
        # scale TFLite = 2^(-fractional_bits) → I = 8 - fractional_bits
        import math
        frac_bits = max(0, -math.floor(math.log2(scale + 1e-12)))
        I = max(1, 8 - frac_bits)
        print(f"{detail['index']:>6}  {detail['name']:45s}  "
              f"{scale:>12.6f}  {zp:>6d}  ap_fixed<8,{I}>")

 Index  Nom                                                   Scale    Zero  ap_fixed suggéré
-------------------------------------------------------------------------------------
     0  input_1_int8                                       0.003922    -128  ap_fixed<8,1>
     1  model/dense/BiasAdd/ReadVariableOp/resource        0.000074       0  ap_fixed<8,1>
     3  model/activation/Relu;model/batch_normalization/FusedBatchNormV3;model/conv2d/BiasAdd/ReadVariableOp/resource;model/conv2d/BiasAdd;model/depthwise_conv2d/depthwise;model/conv2d/Conv2D      0.000064       0  ap_fixed<8,1>
     4  model/activation_1/Relu;model/batch_normalization_1/FusedBatchNormV3;model/depthwise_conv2d/depthwise;model/depthwise_conv2d/BiasAdd;model/depthwise_conv2d/BiasAdd/ReadVariableOp/resource      0.000156       0  ap_fixed<8,1>
     5  model/batch_normalization_1/FusedBatchNormV3;model/depthwise_conv2d/depthwise;model/depthwise_conv2d/BiasAdd;model/depthwise_conv2d/BiasAdd/ReadVariableOp/resource     

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
